# 🚢 Prodigy InfoTech — Data Science Internship
## Task 02: Data Cleaning & Exploratory Data Analysis — Titanic Dataset

**Objective:** Perform data cleaning and EDA on the Titanic dataset. Explore relationships between variables and identify patterns and trends.

**Dataset:** Titanic — Machine Learning from Disaster (Kaggle)  
**Source:** [kaggle.com/c/titanic](https://www.kaggle.com/c/titanic)

**Author:** Sargun (Divsargun Kaur)  
**Internship:** Prodigy InfoTech — Data Science Track, 2026

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 2. Load Dataset

In [ ]:
# Load Titanic dataset (download from Kaggle: https://www.kaggle.com/c/titanic/data)
df = pd.read_csv('titanic.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 3. Initial Exploration

In [ ]:
print('=== Dataset Info ===')
df.info()

print('\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
print('=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

print('\n=== Survival Rate ===')
print(f'Survived: {df["Survived"].sum()} ({df["Survived"].mean():.1%})')
print(f'Did Not Survive: {(df["Survived"]==0).sum()} ({1-df["Survived"].mean():.1%})')

## 4. Data Cleaning

In [ ]:
df_clean = df.copy()

# 1. Fill Age: median grouped by Pclass + Sex (more accurate than global median)
df_clean['Age'].fillna(
    df_clean.groupby(['Pclass', 'Sex'])['Age'].transform('median'),
    inplace=True
)

# 2. Fill Embarked: mode (most common port)
df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0], inplace=True)

# 3. Drop Cabin: >77% missing, too sparse to impute meaningfully
df_clean.drop(columns=['Cabin'], inplace=True)

# 4. Feature Engineering
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'Adult', 'Middle-aged', 'Senior']
)

print('After cleaning:')
print(f'Missing values remaining: {df_clean.isnull().sum().sum()}')
print(f'Shape: {df_clean.shape}')
df_clean.head()

## 5. Exploratory Data Analysis

In [ ]:
# ── Theme ───────────────────────────────────────────────────────────────────
BG      = '#0D1117'
CARD    = '#161B22'
TEXT    = '#E6EDF3'
SUBTEXT = '#8B949E'
GRID    = '#21262D'
ACCENT1 = '#58A6FF'
ACCENT2 = '#F78166'
ACCENT3 = '#3FB950'
ACCENT4 = '#D2A8FF'
GOLD    = '#E3B341'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': GRID, 'axes.labelcolor': TEXT,
    'xtick.color': SUBTEXT, 'ytick.color': SUBTEXT,
    'text.color': TEXT, 'grid.color': GRID, 'grid.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.spines.left': False, 'axes.spines.bottom': False,
})
print('Theme applied ✅')

### 5.1 Overall Survival Rate

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), facecolor=BG)
surv = df_clean['Survived'].value_counts()
bars = ax.bar(['Did Not Survive', 'Survived'], surv.values,
              color=[ACCENT2, ACCENT3], width=0.5, zorder=3)
for bar, val in zip(bars, surv.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+5,
            f'{val} ({val/len(df_clean):.1%})', ha='center', fontsize=11, color=TEXT)
ax.set_title('Overall Survival Rate', fontsize=14, fontweight='bold', color=TEXT, pad=12, loc='left')
ax.set_ylabel('Number of Passengers', fontsize=10, color=SUBTEXT)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart1_survival.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 5.2 Survival by Gender

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), facecolor=BG)
surv_sex = df_clean.groupby('Sex')['Survived'].mean() * 100
bars = ax.bar(surv_sex.index, surv_sex.values, color=[ACCENT1, ACCENT4], width=0.45, zorder=3)
for bar, val in zip(bars, surv_sex.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+1,
            f'{val:.1f}%', ha='center', fontsize=12, color=TEXT, fontweight='bold')
ax.set_title('Survival Rate by Gender', fontsize=14, fontweight='bold', color=TEXT, pad=12, loc='left')
ax.set_ylabel('Survival Rate (%)', fontsize=10, color=SUBTEXT)
ax.set_ylim(0, 100)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart2_gender.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 5.3 Survival by Passenger Class

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), facecolor=BG)
surv_class = df_clean.groupby('Pclass')['Survived'].mean() * 100
bars = ax.bar(['1st Class', '2nd Class', '3rd Class'], surv_class.values,
              color=[GOLD, ACCENT1, ACCENT2], width=0.5, zorder=3)
for bar, val in zip(bars, surv_class.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+1,
            f'{val:.1f}%', ha='center', fontsize=12, color=TEXT, fontweight='bold')
ax.set_title('Survival Rate by Passenger Class', fontsize=14, fontweight='bold', color=TEXT, pad=12, loc='left')
ax.set_ylabel('Survival Rate (%)', fontsize=10, color=SUBTEXT)
ax.set_ylim(0, 100)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart3_pclass.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 5.4 Age Distribution by Survival

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
ax.hist(df_clean[df_clean['Survived']==0]['Age'], bins=25, color=ACCENT2, alpha=0.7, label='Did Not Survive', zorder=3)
ax.hist(df_clean[df_clean['Survived']==1]['Age'], bins=25, color=ACCENT3, alpha=0.7, label='Survived', zorder=3)
ax.axvline(df_clean['Age'].median(), color=GOLD, linewidth=2, linestyle='--',
           label=f'Median Age: {df_clean["Age"].median():.0f}')
ax.set_xlabel('Age', fontsize=10, color=SUBTEXT)
ax.set_ylabel('Count', fontsize=10, color=SUBTEXT)
ax.set_title('Age Distribution by Survival Status', fontsize=14, fontweight='bold', color=TEXT, pad=12, loc='left')
ax.legend(fontsize=9, framealpha=0.15, facecolor=CARD, edgecolor=GRID)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart4_age.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 5.5 Survival by Class & Gender (Grouped Bar)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
pivot = df_clean.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack() * 100
x = np.arange(3)
w = 0.35
b1 = ax.bar(x - w/2, pivot['female'].values, w, color=ACCENT4, label='Female', zorder=3)
b2 = ax.bar(x + w/2, pivot['male'].values,   w, color=ACCENT1, label='Male',   zorder=3)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9, color=TEXT)
ax.set_xticks(x)
ax.set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
ax.set_ylabel('Survival Rate (%)', fontsize=10, color=SUBTEXT)
ax.set_ylim(0, 115)
ax.set_title('Survival Rate by Class & Gender', fontsize=14, fontweight='bold', color=TEXT, pad=12, loc='left')
ax.legend(fontsize=9, framealpha=0.15, facecolor=CARD, edgecolor=GRID)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart5_class_gender.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 5.6 Fare Distribution by Class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), facecolor=BG)
fare_class = [df_clean[df_clean['Pclass']==c]['Fare'].values for c in [1,2,3]]
bp = ax.boxplot(fare_class, patch_artist=True, widths=0.5,
                medianprops=dict(color=TEXT, linewidth=2))
for patch, color in zip(bp['boxes'], [GOLD, ACCENT1, ACCENT2]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for w in bp['whiskers']: w.set_color(SUBTEXT)
for c in bp['caps']: c.set_color(SUBTEXT)
for f in bp['fliers']: f.set_markerfacecolor(SUBTEXT); f.set_markersize(3)
ax.set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
ax.set_ylabel('Fare (£)', fontsize=10, color=SUBTEXT)
ax.set_title('Fare Distribution by Passenger Class', fontsize=14, fontweight='bold', color=TEXT, pad=12, loc='left')
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart6_fare.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 5.7 Survival by Family Size

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
fam_surv = df_clean.groupby('FamilySize')['Survived'].mean() * 100
bar_colors = [ACCENT3 if v > 38 else ACCENT2 for v in fam_surv.values]
bars = ax.bar(fam_surv.index, fam_surv.values, color=bar_colors, width=0.6, zorder=3)
for bar, val in zip(bars, fam_surv.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+1,
            f'{val:.0f}%', ha='center', fontsize=9, color=TEXT)
ax.axhline(38, color=SUBTEXT, linewidth=1.2, linestyle='--', alpha=0.6, label='Overall avg')
ax.set_xlabel('Family Size (self + relatives aboard)', fontsize=10, color=SUBTEXT)
ax.set_ylabel('Survival Rate (%)', fontsize=10, color=SUBTEXT)
ax.set_ylim(0, 100)
ax.set_title('Survival Rate by Family Size', fontsize=14, fontweight='bold', color=TEXT, pad=12, loc='left')
ax.legend(fontsize=9, framealpha=0.15, facecolor=CARD, edgecolor=GRID)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart7_family.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 6. Key Insights

1. **Women survived at a far higher rate than men** — the "women and children first" evacuation protocol is clearly visible in the data.

2. **Passenger class was a strong predictor of survival** — 1st class passengers had the highest survival rate, reflecting both proximity to lifeboats and socioeconomic privilege.

3. **Children had relatively better survival odds** — young age provided some advantage, especially in lower classes where children were prioritized.

4. **Traveling alone was a disadvantage** — small families (2–4 members) had better survival rates than solo travelers or very large families (7+), who struggled to evacuate together.

5. **Fare and class are tightly correlated** — 1st class fares were far higher and far more variable, while 3rd class fares were low and clustered.

6. **Missing data was handled strategically** — Age was imputed using group-specific medians (by class and gender) rather than a global median, preserving demographic accuracy. Cabin was dropped due to >77% missingness.

---
**Dataset:** Titanic — Machine Learning from Disaster | [kaggle.com/c/titanic](https://www.kaggle.com/c/titanic)  
**Author:** Sargun (Divsargun Kaur) | Prodigy InfoTech Data Science Internship 2026